
**🤖 AI Lab Partner Policy: STRICTLY Opt-In Code Generation**

In this course, we treat AI tools (like ChatGPT, Gemini, Copilot) as **Lab Partners**, not solution generators. You must use the following prompt to ensure the AI acts responsibly.

**1. Copy the text inside the block below**
**2. Open your AI Assistant (Gemini, ChatGPT, etc.)**
**3. Paste the text to set the rules for the session**

> "I am a student in an Intro to Machine Learning course. Please act as my **ML Lab Partner**.
> 
> **Your Rules:**
> 
> 1. **Code Generation is STRICTLY Opt-In:** You **MUST NOT** generate any runnable Python code unless my message starts with one of the specific prefixes below (`code:` or `output:`).
>    * *Default Behavior:* If I ask 'How do I...?' or 'Help me with...', explain the strategy in English, provide pseudocode, or use illustrative examples. Do not generate runnable solution code.
> 
> 2. **The 'code:' Trigger (Logic & Calculation):** 
>    * When generating code, prioritize simplicity and human readability. Avoid complex syntax.
>    * **Constraint:** When I use this trigger, provide **only one single line of code**. Do not write full blocks.
> 
> 3. **The 'output:' Trigger (Formatting & Printing):**
>    * Use this ONLY when I request code to print results, format tables, or create plots.
>    * **Exception:** For this trigger only, you **MAY** provide full multi-line code blocks to handle the verbose syntax of formatting or plotting.
> 
> 4. **Wait for Me:** After providing the code, stop immediately. Wait for me to run it and ask for the next step.
> 
> 5. **Explain Briefly:** Add a short comment explaining what the code does.
> 
> 6. **Catch Logic Errors:** If I ask for a step that is methodologically wrong (like testing on training data), stop me and explain the error before proceeding."


# Lecture 12: Overfitting, Regularization & Model Selection

**Topics:**
1. Polynomial regression with `PolynomialFeatures`
2. The overfitting problem
3. Training vs test error
4. Ridge and Lasso regularization
5. Grid search with cross-validation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score

import warnings
warnings.filterwarnings('ignore')  # Suppress convergence warnings for cleaner output

---
## 1. Synthetic Dataset

To study overfitting clearly, we create data from a **known function** with added noise:

$$y = \sin(1.5\pi x) + 0.5x + \text{noise}$$

where the amount of noise is controlled by its standard deviation.

Using synthetic data lets us compare our models against the **true pattern** — something we can't do with real data.

We use a small training set so that overfitting is clearly visible.

In [ ]:
# --- Generate Synthetic Dataset ---
np.random.seed(0)

def true_function(x):
    """The true pattern (unknown in real problems, known here for illustration)."""
    return np.sin(1.5 * np.pi * x) + 0.5 * x

N = 50
x_all = np.random.uniform(0, 1, N)                          # Input feature values; N random points in [0, 1]
y_all = true_function(x_all) + np.random.normal(0, 0.2, N)  # Target values; true pattern + random noise
x_all = x_all.reshape(-1, 1)                                # Reshape to column vector; sklearn requires 2D input

print(f"Total data points: {N}")

### Recall: `train_test_split()`

Randomly splits data into training and test sets:

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=25, random_state=42)
# train_size=25  -> 25 training samples
# random_state=42 -> reproducible random split
```

In [ ]:
# YOUR CODE HERE
# TODO: Split x_all, y_all into train and test sets (25 training points)
x_train, x_test, y_train, y_test = train_test_split(...)
raise NotImplementedError()

print(f"Training: {len(x_train)} points")
print(f"Test: {len(x_test)} points")

x_plot = np.linspace(0, 1, 200)  # 200 evenly spaced points in [0, 1] for plotting smooth curves
x_plot = x_plot.reshape(-1, 1)   # Reshape to column vector; sklearn requires 2D input

plt.figure(figsize=(10, 6))
plt.scatter(x_train, y_train, c='blue', s=50, zorder=5, label=f'Train (N={len(x_train)})')
plt.scatter(x_test, y_test, c='gray', alpha=0.3, label=f'Test (N={len(x_test)})')
plt.plot(x_plot, true_function(x_plot), 'k--', linewidth=1.5, label='True function')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Synthetic Dataset: Train vs Test')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 2. Polynomial Features with sklearn

In L11, we built polynomial feature matrices manually:

```python
X_poly = np.column_stack([x**p for p in range(M + 1)])  # Manual approach
```

sklearn provides `PolynomialFeatures` which does this automatically — and handles multiple input variables too.

### NEW: `PolynomialFeatures()`

```python
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=3)      # Create transformer
X_poly = poly.fit_transform(x)           # Transforms x -> [1, x, x^2, x^3]
# Shape: if x has shape (N, 1), X_poly has shape (N, 4)
```

**`fit_transform` vs `transform`:** Unlike `StandardScaler` (which *learns* mean and std from the data), `PolynomialFeatures` doesn't learn anything data-dependent — it just needs to see the input shape once. We still use `fit_transform()` on the first call and `transform()` on subsequent data because that's how sklearn's transformer API works.

In [ ]:
x_demo = np.array([2, 5, 10]).reshape(-1, 1)

poly_demo = PolynomialFeatures(degree=3)
X_demo = poly_demo.fit_transform(x_demo)  # x -> [1, x, x^2, x^3]

print(f"Input shape:  {x_demo.shape}")
print(f"Output shape: {X_demo.shape}")
print(f"Feature names: {poly_demo.get_feature_names_out()}")
print(f"\nInput x = {x_demo.flatten()}")
print(f"Output:\n{X_demo}")

---
## 3. The Effect of Polynomial Degree

### Recall: `StandardScaler()`

Standardizes features to zero mean and unit variance. Use `fit_transform()` on training data, then `transform()` on test/plot data:

```python
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # Compute mean/std from training data and apply
X_test_scaled = scaler.transform(X_test)          # Apply same mean/std (no re-fitting!)
```

Use `PolynomialFeatures`, `StandardScaler`, and `LinearRegression` to fit polynomial models of different degrees to the **training data**. Plot the results alongside the true function.

In [ ]:
degrees = [1, 3, 8]
colors = ['red', 'green', 'orange']

plt.figure(figsize=(12, 7))
plt.scatter(x_train, y_train, c='blue', s=50, zorder=5, label='Training data')
plt.plot(x_plot, true_function(x_plot), 'k--', linewidth=1.5, alpha=0.5, label='True function')

for i in range(len(degrees)):
    M = degrees[i]
    color = colors[i]

    # YOUR CODE HERE
    # TODO: Create degree-M polynomial features from x_train and x_plot
    poly = PolynomialFeatures(degree=...)
    X_train_poly = poly.fit_transform(...)
    X_plot_poly = poly.transform(...)
    #
    # TODO: Standardize the polynomial features
    scaler = StandardScaler()
    X_train_poly = scaler.fit_transform(...)  # Fit on training features
    X_plot_poly = scaler.transform(...)        # Apply same stats to plot grid
    #
    # TODO: Fit a LinearRegression model on the training data
    model = LinearRegression()
    model.fit(...)
    #
    # TODO: Predict y values on the plotting grid
    y_plot = model.predict(...)
    raise NotImplementedError()

    plt.plot(x_plot, y_plot, color=color, linewidth=2, label=f'Degree {M}')

plt.xlabel('x')
plt.ylabel('y')
plt.title('Polynomial Regression: Effect of Degree M')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
"""Check"""
assert X_train_poly.shape[1] == M + 1, f"Expected {M+1} features, got {X_train_poly.shape[1]}"
print(f"Degree {M} feature matrix shape: {X_train_poly.shape}")
print("Check passed!")

**Observation:**
- **Low degree** (linear): Captures the upward trend but misses the curvature — **underfitting**
- **Moderate degree**: Good fit — captures the sine-like shape
- **High degree**: Starts to wiggle, chasing individual training points — **overfitting**

---
## 4. Training vs Test Error

How do we *measure* overfitting? Compare the error on training data vs unseen test data.

**Key idea:** Training error always decreases with model complexity, but test error follows a U-shape.


For each polynomial degree $M = 1, 2, \ldots, 15$:
1. Fit a polynomial model on the training data
2. Compute the Root Mean Squared Error (RMSE) on both train and test:

$$\text{RMSE} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2}$$

In [ ]:
max_degree = 15
train_errors = []
test_errors = []

for M in range(1, max_degree + 1):

    # YOUR CODE HERE
    # TODO: Create degree-M polynomial features from x_train and x_test
    poly = PolynomialFeatures(degree=...)
    X_train_poly = poly.fit_transform(...)
    X_test_poly = poly.transform(...)
    #
    # TODO: Standardize the polynomial features
    scaler = StandardScaler()
    X_train_poly = scaler.fit_transform(...)  # Fit on training features
    X_test_poly = scaler.transform(...)        # Apply training stats to test
    #
    model = LinearRegression()
    model.fit(...)
    #
    # TODO: Compute RMSE on train and test
    y_train_pred = model.predict(...)
    y_test_pred = model.predict(...)
    rmse_train = np.sqrt(np.mean((...)**2))
    rmse_test = np.sqrt(np.mean((...)**2))
    raise NotImplementedError()

    train_errors.append(rmse_train)
    test_errors.append(rmse_test)

print(f"{'M':<5} {'Train RMSE':<15} {'Test RMSE':<15}")
print("-" * 35)
for M in range(1, max_degree + 1):
    print(f"{M:<5} {train_errors[M-1]:<15.4f} {test_errors[M-1]:<15.4f}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, max_degree + 1), train_errors, 'bo-', linewidth=2,
         markersize=6, label='Training Error')
plt.plot(range(1, max_degree + 1), test_errors, 'ro-', linewidth=2,
         markersize=6, label='Test Error')

best_M = np.argmin(test_errors) + 1  # +1 because degrees start at 1, not 0
plt.axvline(x=best_M, color='green', linestyle='--', linewidth=2,
            label=f'Best M = {best_M}')

plt.xlabel('Polynomial Degree (M)')
plt.ylabel('RMSE')
plt.title('Training vs Test Error')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.xticks(range(1, max_degree + 1))
plt.show()

print(f"Best degree by test error: M = {best_M}")

In [ ]:
"""Check"""
assert len(train_errors) == max_degree, f"Expected {max_degree} errors, got {len(train_errors)}"
assert train_errors[-1] < train_errors[0], "Training error should decrease with degree"
print("Check passed!")

**Key observations:**
1. **Training error always decreases** as $M$ increases
2. **Test error decreases then increases** (U-shape) — past the optimal $M$, the model overfits
3. The **gap** between train and test error grows with $M$ — this gap signals overfitting

---
## 5. Why Overfitting Happens: Coefficient Instability

When the polynomial degree is too high, the model fits noise by using **large, oscillating coefficients**.

In [ ]:
for M in [2, 5, 10, 14]:
    poly = PolynomialFeatures(degree=M)
    X_poly = poly.fit_transform(x_train)    # x -> [1, x, ..., x^M]

    scaler = StandardScaler()
    X_poly = scaler.fit_transform(X_poly)   # Standardize so coefficients are comparable across degrees

    model = LinearRegression()
    model.fit(X_poly, y_train)

    max_coef = np.max(np.abs(model.coef_))  # Largest coefficient magnitude; shows instability with higher degrees
    print(f"Degree {M:>2d}: max |coefficient| = {max_coef:.2e}")

**Observation:** As the degree increases, coefficients grow by many orders of magnitude. A coefficient of $10^{10}$ or more means tiny changes in $x$ cause enormous swings in $\hat{y}$.

**The core idea:** We need a way to *penalize* large coefficients. This is **regularization**.

---
## 6. Ridge Regression (L2 Regularization)

Standard linear regression minimizes:

$$\text{SSE} = \sum (y_i - \hat{y}_i)^2$$

**Ridge regression** adds a penalty on the coefficient magnitudes:

$$\text{Cost} = \sum (y_i - \hat{y}_i)^2 + \lambda \sum a_j^2$$

- $\lambda = 0$: standard regression (no penalty)
- $\lambda \to \infty$: all coefficients shrink to zero

### NEW: `Ridge()`

```python
from sklearn.linear_model import Ridge

model = Ridge(alpha=0.1)       # alpha is sklearn's name for lambda
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
```

**Note:** sklearn calls the regularization parameter `alpha` instead of $\lambda$.

**Important:** Before using Ridge, standardize the polynomial features with `StandardScaler` so the penalty treats all features equally.

Using a high-degree polynomial ($M = 12$):
1. Create polynomial features with `PolynomialFeatures`
2. Standardize the features (using training mean and std)
3. Fit `Ridge` models with $\lambda \in \{0, 0.01, 1, 100\}$
4. Plot the results

In [ ]:
M = 12
lambda_values = [0, 0.01, 1, 100]
colors = ['red', 'orange', 'green', 'blue']

poly = PolynomialFeatures(degree=M)
X_train_poly = poly.fit_transform(x_train)  # x -> [1, x, ..., x^M]
X_test_poly = poly.transform(x_test)
X_plot_poly = poly.transform(x_plot)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_poly)  # Learn mean/std from training and apply
X_test_scaled = scaler.transform(X_test_poly)         # Apply training stats to test data
X_plot_scaled = scaler.transform(X_plot_poly)         # Apply training stats to plot grid

plt.figure(figsize=(12, 7))
plt.scatter(x_train, y_train, c='blue', s=50, zorder=5, label='Training data')
plt.plot(x_plot, true_function(x_plot), 'k--', linewidth=1.5, alpha=0.5, label='True function')

for i in range(len(lambda_values)):
    lam = lambda_values[i]
    color = colors[i]

    # YOUR CODE HERE
    # TODO: Fit Ridge model with given alpha (lambda)
    model = Ridge(alpha=...)
    model.fit(...)
    #
    # TODO: Predict on the plotting grid
    y_plot = model.predict(...)
    raise NotImplementedError()

    plt.plot(x_plot, y_plot, color=color, linewidth=2, label=f'lambda = {lam}')

plt.xlabel('x')
plt.ylabel('y')
plt.title(f'Ridge Regression (M={M}): Effect of Lambda')
plt.ylim(-1, 1.5)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
"""Check"""
n_features = model.coef_.shape[0]
assert n_features == X_train_scaled.shape[1], f"Expected {X_train_scaled.shape[1]} features, got {n_features}"
print(f"Max coefficient with lambda={lambda_values[-1]}: {np.max(np.abs(model.coef_)):.4f}")
print("Check passed!")

**Observation:**
- **No penalty**: Same as ordinary regression — overfits with wild oscillations
- **Small penalty**: Slight smoothing
- **Moderate penalty**: Good fit — captures the trend without overfitting
- **Large penalty**: Too much regularization — nearly a flat line (underfitting)

Ridge lets us use a high-degree polynomial while keeping coefficients under control!

---
## 7. Lasso Regression (L1 Regularization)

**Ridge** shrinks all coefficients toward zero but keeps them non-zero.

**Lasso** uses a different penalty that can set coefficients **exactly to zero** — automatic feature selection!

$$\text{Cost}_{\text{Lasso}} = \sum (y_i - \hat{y}_i)^2 + \lambda \sum |a_j|$$

### NEW: `Lasso()`

```python
from sklearn.linear_model import Lasso

model = Lasso(alpha=0.1)
model.fit(X_train, y_train)
# model.coef_ may contain exact zeros!
```

Fit both Ridge and Lasso on degree-12 polynomial features (already standardized above) with the same $\lambda$. Compare their coefficients using a bar chart.

In [ ]:
lam = 1

# YOUR CODE HERE
# TODO: Fit Ridge and Lasso with the same alpha
ridge_model = Ridge(alpha=...)
ridge_model.fit(...)
#
lasso_model = Lasso(alpha=...)
lasso_model.fit(...)
raise NotImplementedError()

n_features = len(ridge_model.coef_)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(n_features), ridge_model.coef_, color='steelblue')
axes[0].set_xlabel('Feature Index (power of x)')
axes[0].set_ylabel('Coefficient Value')
axes[0].set_title(f'Ridge (lambda={lam})')
axes[0].axhline(y=0, color='black', linewidth=0.5)

axes[1].bar(range(n_features), lasso_model.coef_, color='coral')
axes[1].set_xlabel('Feature Index (power of x)')
axes[1].set_ylabel('Coefficient Value')
n_zeros = np.sum(np.abs(lasso_model.coef_) < 1e-10)
axes[1].set_title(f'Lasso (lambda={lam}): {n_zeros} coefficients = 0')
axes[1].axhline(y=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

print(f"Ridge: {np.sum(ridge_model.coef_ == 0)} coefficients exactly zero")
print(f"Lasso: {n_zeros} coefficients effectively zero")

**Key difference:**
- **Ridge (L2)**: Shrinks all coefficients toward zero, but keeps them non-zero
- **Lasso (L1)**: Can set coefficients exactly to zero — automatic feature selection

**When to use which?**
- **Ridge**: When you believe all features contribute
- **Lasso**: When you suspect only a few features matter

---
## 8. Grid Search with Cross-Validation

So far we've been choosing $M$ and $\lambda$ by looking at a single test set. But with small data, this estimate is noisy.

**K-Fold Cross-Validation** gives a more reliable estimate by rotating through different train/validation splits.

**Grid search** tries every combination of hyperparameters and picks the one with the lowest CV error.

### Recall: `cross_val_score()`

```python
model = Lasso(alpha=0.1, max_iter=10000)
# 5-fold CV: returns array of 5 scores
scores = cross_val_score(model, X_train, y_train, cv=5,
                         scoring='neg_root_mean_squared_error')
mean_rmse = -scores.mean()   # Negate because sklearn uses negative scores
```

**Why negative?** sklearn convention: higher scores = better, so error metrics are negated.

Use 5-fold cross-validation to search over a grid of polynomial degrees and regularization strengths for Lasso:

- Degrees: $M = 1, 2, \ldots, 10$
- Regularization: $\lambda \in \{0.001, 0.01, 0.1, 1\}$

For each $(M, \lambda)$ combination, compute the mean CV RMSE and track the best.

In [ ]:
degree_range = range(1, 11)
lambda_values = [0.001, 0.01, 0.1, 1]

best_rmse = float('inf')
best_M_cv = None
best_lambda_cv = None
results = {}

for lam in lambda_values:
    cv_errors = []
    for M in degree_range:

        # YOUR CODE HERE
        # TODO: Create degree-M polynomial features from x_train and standardize
        poly = PolynomialFeatures(degree=...)
        X_poly = poly.fit_transform(...)
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(...)
        #
        # TODO: Run 5-fold cross-validation with Lasso
        scores = cross_val_score(Lasso(alpha=lam, max_iter=10000), X_scaled, y_train,
                                 cv=..., scoring='neg_root_mean_squared_error')
        mean_rmse = ...
        raise NotImplementedError()

        cv_errors.append(mean_rmse)
        if mean_rmse < best_rmse:
            best_rmse = mean_rmse
            best_M_cv = M
            best_lambda_cv = lam

    results[lam] = cv_errors

print(f"Best: M={best_M_cv}, lambda={best_lambda_cv}, CV RMSE={best_rmse:.4f}")

In [ ]:
plt.figure(figsize=(10, 6))

for lam in lambda_values:
    plt.plot(list(degree_range), results[lam], 'o-', linewidth=2, markersize=6,
             label=f'lambda={lam}')

plt.axvline(x=best_M_cv, color='green', linestyle='--', linewidth=1, alpha=0.5)

plt.xlabel('Polynomial Degree (M)')
plt.ylabel('Cross-Validation RMSE')
plt.title('Grid Search: Lasso CV Error by Degree and Lambda')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(list(degree_range))
plt.show()

print(f"\nBest: M={best_M_cv}, lambda={best_lambda_cv}, CV RMSE={best_rmse:.4f}")

In [ ]:
"""Check"""
print(f"Best M = {best_M_cv}, Best lambda = {best_lambda_cv}")
print(f"CV RMSE = {best_rmse:.4f}")
print("Check passed!")

---
## 9. Putting It Together: Final Model

Now let's use our grid search result to build and evaluate the final model.

In [ ]:
# --- Build final model using the best hyperparameters from grid search ---
poly_final = PolynomialFeatures(degree=best_M_cv)
X_train_final = poly_final.fit_transform(x_train)
X_test_final = poly_final.transform(x_test)

scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train_final)
X_test_std = scaler.transform(X_test_final)

model_final = Lasso(alpha=best_lambda_cv, max_iter=10000)
model_final.fit(X_train_std, y_train)

y_test_pred = model_final.predict(X_test_std)  # Evaluate on test set (only done ONCE, after all tuning)
test_rmse = np.sqrt(np.mean((y_test - y_test_pred)**2))

print(f"Final model: Degree {best_M_cv}, Lasso (lambda={best_lambda_cv})")
print(f"Test RMSE: {test_rmse:.4f}")

X_plot_final = poly_final.transform(x_plot)
X_plot_std = scaler.transform(X_plot_final)
y_plot_final = model_final.predict(X_plot_std)

plt.figure(figsize=(10, 6))
plt.scatter(x_train, y_train, c='blue', s=50, zorder=5, label=f'Train (N={len(x_train)})')
plt.scatter(x_test, y_test, c='gray', alpha=0.3, label=f'Test (N={len(x_test)})')
plt.plot(x_plot, true_function(x_plot), 'k--', linewidth=1.5, label='True function')
plt.plot(x_plot, y_plot_final, 'r-', linewidth=2, label=f'Final model (M={best_M_cv})')
plt.xlabel('x')
plt.ylabel('y')
plt.title(f'Final Model: Degree {best_M_cv} + Lasso (lambda={best_lambda_cv})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## Summary

1. **`PolynomialFeatures`**: sklearn tool that builds $[1, x, x^2, \ldots, x^M]$ automatically
2. **Overfitting**: High-degree polynomials fit training data well but fail on new data — large, unstable coefficients
3. **Training vs test error**: Training error always decreases; test error follows a U-shape
4. **Ridge (L2)**: Penalizes $\sum a_j^2$ — shrinks all coefficients toward zero
5. **Lasso (L1)**: Penalizes $\sum |a_j|$ — can zero out coefficients (feature selection)
6. **Cross-validation**: Rotate through train/validation splits for reliable error estimates
7. **Grid search**: Try every combination of hyperparameters ($M$, $\lambda$), pick the one with the lowest CV error, evaluate **once** on test set